In [12]:
from pathlib import Path
from tabulate import tabulate
import re

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr

In [13]:
project_dir = Path.cwd().parent

# Check input population data

# ORIGINAL

In [14]:
# read in orginal data from dirs (tif files) and make xarray datasets
dir_2UP_orginal = r"Z:\cold_data_storage\users\roelfsemam\surdrive_UU\NSA\Downscaling_share\input_data_downscaling\Population\processed\2UP\processed_GHSL_2024_M3\SSP2"
dir_COMASS_orginal = r"Z:\cold_data_storage\users\roelfsemam\surdrive_UU\NSA\Downscaling_share\input_data_downscaling\Population\processed\COMPASS\processed_version_2\SSP2"
dirs = [dir_2UP_orginal, dir_COMASS_orginal ]

In [15]:
search_pattern_2UP = rf"M?var1_SSP2_(\d{{4}})_tpop\.tif"
search_pattern_COMPASS = search_pattern = f"Population_count_(\\d{{4}})_6min_SSP2_not_harm.tif"
search_patterns = [search_pattern_2UP, search_pattern_COMPASS]

sources = ["2UP", "COMPASS"]

In [ ]:
df_pop_totals = pd.DataFrame()
for i, d in enumerate(dirs):
    files = list(Path(d).glob("*.tif"))
    data_list, years, opened = [], [], []
    for j, f in enumerate(files):
        print(f"Reading file: {f.name}")
        try:
            year = int(re.search(search_patterns[i], f.name).group(1))
        except AttributeError:
            print(f"Could not extract year from file: {f.name}")
            continue
        years.append(year)
        da = rxr.open_rasterio(f, chunks={"x": "auto", "y": "auto"})

        # Set/update profile parameters
        crs = CRS.from_epsg(4326)
        cell_size = (0.5/60)
        x_min = -180.0
        y_max = 90.0
        transform = Affine(
            cell_size, 0.0, x_min,
            0.0, -cell_size, y_max
        )

        profile.update({
            "crs": crs,
            "transform": transform,
            "nodata": -9999
        })

        opened.append(da)  # keep the raw handle so it can be closed in the finally block
        da = da.squeeze(drop=True)
        da = da.chunk("auto")
        data_list.append(da)
    xr_pop = xr.concat(data_list, dim="time")
    xr_pop = xr_pop.assign_coords(time=years)
    xr_pop = xr_pop.to_dataset(name="Population")
    xr_pop = xr_pop.sortby("y", ascending=False)  # north-to-south
    xr_pop = xr_pop.sortby("x", ascending=True)  # west-to-east
    xr_pop_total = xr_pop["Population"].sum(dim=["y", "x"])
    df_pop_total = xr_pop_total.to_dataframe().reset_index()
    df_pop_total["source"] = sources[i]
    df_pop_totals = pd.concat([df_pop_totals, df_pop_total], ignore_index=True)
df_pop_totals = df_pop_totals.pivot(index="time", columns="source", values="Population").reset_index()
df_pop_totals.to_csv(project_dir / "data" / "check" / f"check_population_original.csv", index=True, float_format="%.1f", sep=";")
df_pop_totals


Reading file: M3var1_SSP2_2010_tpop.tif
Reading file: M3var1_SSP2_2015_tpop.tif
Reading file: M3var1_SSP2_2020_tpop.tif
Reading file: M3var1_SSP2_2025_tpop.tif
Reading file: M3var1_SSP2_2030_tpop.tif
Reading file: M3var1_SSP2_2035_tpop.tif
Reading file: M3var1_SSP2_2040_tpop.tif
Reading file: M3var1_SSP2_2045_tpop.tif
Reading file: M3var1_SSP2_2050_tpop.tif
Reading file: M3var1_SSP2_2055_tpop.tif
Reading file: M3var1_SSP2_2060_tpop.tif
Reading file: M3var1_SSP2_2065_tpop.tif
Reading file: M3var1_SSP2_2070_tpop.tif
Reading file: M3var1_SSP2_2075_tpop.tif
Reading file: M3var1_SSP2_2080_tpop.tif
Reading file: M3var1_SSP2_2085_tpop.tif
Reading file: M3var1_SSP2_2090_tpop.tif
Reading file: M3var1_SSP2_2095_tpop.tif
Reading file: M3var1_SSP2_2100_tpop.tif
Reading file: Population_count_2015_6min_SSP2_not_harm.tif
Reading file: Population_count_2020_6min_SSP2_not_harm.tif
Reading file: Population_count_2025_6min_SSP2_not_harm.tif
Reading file: Population_count_2030_6min_SSP2_not_harm.tif
Read

source,time,2UP,COMPASS
0,2010,-7.280622e+12,NaN
1,2015,-7.280181e+12,7.470465e+09
2,2020,-7.279769e+12,7.886975e+09
3,2025,-7.279420e+12,8.240891e+09
4,2030,-7.279075e+12,8.593658e+09
5,2035,-7.278755e+12,8.921750e+09
6,2040,-7.278463e+12,9.223152e+09
7,2045,-7.278207e+12,9.491279e+09
8,2050,-7.277990e+12,9.721124e+09
9,2055,-7.277815e+12,9.908384e+09


## PROCESSED

In [3]:
dir_2UP_processed = "K:/PythonWork/downscaling/Kaya_downscaling/data/processed/base_run"
dir_COMPAS_processed = "K:/PythonWork/downscaling/Kaya_downscaling/data/processed/sensitivity_4"
file_2UP_processed = "Population_2UP_GHSL_2024_M3_SSP2_cf_12.nc"
file_COMPASS_processed = "Population_COMPASS_version_2_SSP2_cf_1.nc"

In [4]:
xr_pop_2UP_processed = xr.open_dataset(Path(dir_2UP_processed) / file_2UP_processed)
xr_pop_COMPASS_processed = xr.open_dataset(Path(dir_COMPAS_processed) / file_COMPASS_processed)

In [5]:
xr_pop_2UP_processed_total = xr_pop_2UP_processed["Population"].sum(dim=["y", "x"])
df_pop_2UP_processed_total = xr_pop_2UP_processed_total.to_dataframe().reset_index()
df_pop_2UP_processed_total["source"] = "2UP"
df_pop_2UP_processed_total.rename(columns={"time": "year"}, inplace=True)
df_pop_2UP_processed_total.drop(columns=["spatial_ref"], inplace=True, errors="ignore")
xr_pop_COMPASS_processed_total = xr_pop_COMPASS_processed["Population"].sum(dim=["y", "x"])
df_pop_COMPASS_processed_total = xr_pop_COMPASS_processed_total.to_dataframe().reset_index()
df_pop_COMPASS_processed_total["source"] = "COMPASS"
df_pop_COMPASS_processed_total.rename(columns={"time": "year"}, inplace=True)
df_pop_COMPASS_processed_total.drop(columns=["spatial_ref"], inplace=True, errors="ignore")

In [8]:
df_pop_processed = pd.concat([df_pop_2UP_processed_total, df_pop_COMPASS_processed_total], ignore_index=True)
df_pop_processed = df_pop_processed.pivot(index="year", columns="source", values="Population").reset_index()
df_pop_processed.to_csv(project_dir / "data" / "check" / f"check_population.csv", index=True, float_format="%.1f", sep=";")
df_pop_processed

source,year,2UP,COMPASS
0,2010,6.983693e+09,NaN
1,2015,7.423233e+09,7.470465e+09
2,2020,7.836372e+09,7.886975e+09
3,2025,8.185335e+09,8.240891e+09
4,2030,8.530188e+09,8.593658e+09
5,2035,8.849691e+09,8.921750e+09
6,2040,9.141745e+09,9.223152e+09
7,2045,9.398523e+09,9.491279e+09
8,2050,9.615189e+09,9.721124e+09
9,2055,9.790235e+09,9.908384e+09
